# Factors model

Cross-sectional ranking of the universe's **stocks** on 5 factors. The scores are not a forecast: they rank each
stock against its peers today.

| Factor | Metrics (higher = better) |
|---|---|
| Momentum | 12-month price change, distance above the 200-day average |
| Quality | ROA, operating margin, −debt/equity, FCF margin |
| Value | Forward E/P, B/P, S/P, EBITDA/EV |
| Growth | Revenue growth, earnings growth, expected EPS growth |
| Size | log(market cap) |

Method:
1. Each metric is winsorized (1st/99th percentile) and z-scored within its **(market, sector)** cell.
2. A factor is the average of its available metric z-scores (1/n), re-z-scored within the same cell.
3. Size is log(market cap) z-scored within the **market** only.
4. A stock with no data for a factor gets 0 (neutral).

The most positive score is the best stock of its cell.

In [1]:
import io
import json
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import yfinance as yf

## 1. Universe (S3)

In [2]:
BUCKET = "monport-datasets-381491955506"
KEY = "universe/universe.parquet"

# boto3 instead of pd.read_parquet("s3://..."): pyarrow's S3 client doesn't always find the same credentials.
obj = boto3.client("s3", region_name="us-east-1").get_object(Bucket=BUCKET, Key=KEY)
universe = pd.read_parquet(io.BytesIO(obj["Body"].read()))
print(universe.shape, universe["symbol"].nunique(), "symbols")

/workshop/.venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


(1167293, 20) 804 symbols


Only stocks get factor scores. Cells use the **listing market** (US, Canada, France) instead of the country of
domicile: several domicile countries have 1 to 4 stocks (Luxembourg, Singapore, Australia, Bermuda), too few to
z-score, and the market matches the currency of the ratios.

In [3]:
stocks = (universe[universe["instrument_type"] == "Stock"]
          .drop_duplicates("symbol")[["symbol", "asset_class", "instrument_type", "sector", "country"]]
          .reset_index(drop=True))

def listing_market(symbol):
    if symbol.endswith(".TO"):
        return "Canada"
    if symbol.endswith(".PA"):
        return "France"
    return "United States"

stocks["market"] = stocks["symbol"].map(listing_market)
print(len(stocks), "stocks")
stocks.groupby(["market", "sector"]).size().unstack(0)

721 stocks


market,Canada,France,United States
sector,,,
Communication Services,4,2,21
Consumer Discretionary,10,7,47
Consumer Staples,8,4,29
Energy,28,1,21
Financials,21,5,75
Health Care,2,3,60
Industrials,23,10,78
Information Technology,8,3,84
Materials,50,1,26


## 2. Fundamentals (yfinance `Ticker().info`)

One request per ticker. Responses are cached in `data/factors_info_cache.json`: a rerun only fetches missing tickers.
Delete the file to refresh everything.

In [4]:
INFO_FIELDS = [
    "marketCap", "fiftyTwoWeekChangePercent", "twoHundredDayAverageChangePercent",
    "returnOnAssets", "operatingMargins", "debtToEquity", "freeCashflow", "totalRevenue",
    "forwardPE", "priceToBook", "priceToSalesTrailing12Months", "enterpriseToEbitda",
    "revenueGrowth", "earningsGrowth", "forwardEps", "trailingEps",
]
CACHE = Path("data/factors_info_cache.json")

cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
todo = [s for s in stocks["symbol"] if s not in cache]
print(len(cache), "cached |", len(todo), "to fetch")

for i, symbol in enumerate(todo, 1):
    try:
        info = yf.Ticker(symbol).info
    except Exception as e:
        print(symbol, "failed:", e)
        continue
    cache[symbol] = {field: info.get(field) for field in INFO_FIELDS}
    if i % 50 == 0 or i == len(todo):
        CACHE.write_text(json.dumps(cache))
        print(i, "/", len(todo))

721 cached | 0 to fetch


In [5]:
info = pd.DataFrame.from_dict(cache, orient="index").reindex(stocks["symbol"])
info = info.apply(pd.to_numeric, errors="coerce")
df = stocks.join(info, on="symbol")
print("missing per field:")
df[INFO_FIELDS].isna().sum()

missing per field:


marketCap                             3
fiftyTwoWeekChangePercent             0
twoHundredDayAverageChangePercent     0
returnOnAssets                        5
operatingMargins                      2
debtToEquity                         72
freeCashflow                         45
totalRevenue                          9
forwardPE                             2
priceToBook                           0
priceToSalesTrailing12Months         12
enterpriseToEbitda                   49
revenueGrowth                         9
earningsGrowth                       80
forwardEps                            4
trailingEps                           2
dtype: int64

## 3. Raw metrics

All metrics are oriented so that **higher = better**:
- Ratios where low is good are inverted (P/B → B/P) or sign-flipped (−D/E).
- A negative E/P or B/P (losses, negative equity) is kept: it correctly ranks as expensive.
- Expected EPS growth is only computed when trailing EPS is positive (otherwise the ratio is meaningless).
- Negative-equity companies (MO, MCD, VRSN…) have no D/E in Yahoo: they get the worst leverage value.

In [6]:
def inverse(x):
    return 1 / x.where(x != 0)

# Momentum
df["mom_12m"] = df["fiftyTwoWeekChangePercent"]
df["mom_200d"] = df["twoHundredDayAverageChangePercent"]

# Quality
df["qual_roa"] = df["returnOnAssets"]
df["qual_op_margin"] = df["operatingMargins"]
df["qual_neg_de"] = -df["debtToEquity"]
# Negative equity (P/B < 0): Yahoo leaves D/E blank. These are the most leveraged firms, not unknowns,
# so they get the worst observed value instead of being skipped (which would reward them).
negative_equity = df["debtToEquity"].isna() & (df["priceToBook"] < 0)
df.loc[negative_equity, "qual_neg_de"] = df["qual_neg_de"].min()
print(negative_equity.sum(), "negative-equity stocks get the worst leverage:", ", ".join(df.loc[negative_equity, "symbol"].head(8)), "...")
df["qual_fcf_margin"] = df["freeCashflow"] / df["totalRevenue"].where(df["totalRevenue"] > 0)

# Value
df["val_fwd_ep"] = inverse(df["forwardPE"])
df["val_bp"] = inverse(df["priceToBook"])
df["val_sp"] = inverse(df["priceToSalesTrailing12Months"])
df["val_ebitda_ev"] = inverse(df["enterpriseToEbitda"])

# Growth
df["gro_revenue"] = df["revenueGrowth"]
df["gro_earnings"] = df["earningsGrowth"]
df["gro_exp_eps"] = (df["forwardEps"] / df["trailingEps"] - 1).where(df["trailingEps"] > 0)

# Size
df["size_log_mcap"] = np.log(df["marketCap"].where(df["marketCap"] > 0))

36 negative-equity stocks get the worst leverage: ABBV, AZO, BBD-B.TO, BHC.TO, BKNG, CAH, CCI, DELL ...


In [7]:
FACTORS = {
    "momentum": ["mom_12m", "mom_200d"],
    "quality": ["qual_roa", "qual_op_margin", "qual_neg_de", "qual_fcf_margin"],
    "value": ["val_fwd_ep", "val_bp", "val_sp", "val_ebitda_ev"],
    "growth": ["gro_revenue", "gro_earnings", "gro_exp_eps"],
}
METRICS = [m for metrics in FACTORS.values() for m in metrics]

coverage = df[METRICS + ["size_log_mcap"]].notna().mean().round(3)
coverage.to_frame("coverage")

,coverage
mom_12m,1.000
mom_200d,1.000
qual_roa,0.993
qual_op_margin,0.997
qual_neg_de,0.950
qual_fcf_margin,0.928
val_fwd_ep,0.997
val_bp,1.000
val_sp,0.983
val_ebitda_ev,0.932


## 4. Z-scores by cell

- **Winsorize** each metric at the 1st and 99th percentiles of all stocks, so a few extreme values (e.g. tiny
  revenue bases) don't dominate a cell.
- **Z-score** within the (market, sector) cell. A cell with fewer than `MIN_CELL` values for that metric falls
  back to the market as a whole.

In [8]:
CELL = ["market", "sector"]
FALLBACK = ["market"]
MIN_CELL = 5
WINSOR = (0.01, 0.99)

def winsorize(x):
    lo, hi = x.quantile(WINSOR[0]), x.quantile(WINSOR[1])
    return x.clip(lo, hi)

def zscore(x):
    std = x.std()
    if x.count() < 2 or not std > 0:
        return x * np.nan
    return (x - x.mean()) / std

def cell_zscore(data, col, cell=CELL, fallback=FALLBACK):
    n = data.groupby(cell)[col].transform("count")
    z_cell = data.groupby(cell)[col].transform(zscore)
    z_fallback = data.groupby(fallback)[col].transform(zscore)
    return z_cell.where(n >= MIN_CELL, z_fallback)

In [9]:
for metric in METRICS:
    df[metric] = winsorize(df[metric])
    df[f"{metric}_z"] = cell_zscore(df, metric)

df[[f"{m}_z" for m in METRICS]].describe().T[["count", "mean", "std", "min", "max"]].round(2)

,count,mean,std,min,max
mom_12m_z,721.0,0.01,0.99,-3.19,3.64
mom_200d_z,721.0,0.01,0.99,-3.58,2.79
qual_roa_z,716.0,-0.00,0.97,-2.33,4.24
qual_op_margin_z,719.0,-0.00,0.98,-2.80,3.37
qual_neg_de_z,685.0,-0.01,1.03,-8.87,1.56
qual_fcf_margin_z,669.0,0.00,0.98,-4.98,4.70
val_fwd_ep_z,719.0,0.00,0.99,-2.83,3.81
val_bp_z,721.0,-0.00,0.98,-2.29,3.91
val_sp_z,709.0,0.00,0.98,-1.26,6.06
val_ebitda_ev_z,672.0,0.01,0.98,-2.65,3.95


## 5. Factor scores

Each factor = 1/n of its available metric z-scores (a stock missing a metric is averaged over the others), then
re-z-scored within the (market, sector) cell. A stock with no metric at all for a factor gets 0.

In [10]:
for factor, metrics in FACTORS.items():
    df[f"{factor}_raw"] = df[[f"{m}_z" for m in metrics]].mean(axis=1)
    df[factor] = cell_zscore(df, f"{factor}_raw").fillna(0)

df[list(FACTORS)].describe().round(2)

,momentum,quality,value,growth
count,721.00,721.00,721.00,721.00
mean,0.01,-0.00,0.00,-0.00
std,0.99,0.98,0.98,0.98
min,-3.66,-3.73,-2.20,-3.07
25%,-0.58,-0.57,-0.68,-0.52
50%,0.03,-0.07,-0.16,-0.23
75%,0.63,0.58,0.57,0.26
max,2.86,3.52,5.20,5.78


## 6. Size

log(market cap) z-scored within the listing market. Larger company = higher score.

In [11]:
df["size"] = df.groupby("market")["size_log_mcap"].transform(zscore).fillna(0)
df[["symbol", "market", "marketCap", "size"]].sort_values("size", ascending=False).head(10)

,symbol,market,marketCap,size
1,AAPL,United States,5.177380e+12,4.014922
469,NVDA,United States,3.932339e+12,3.773170
442,MSFT,United States,3.890435e+12,3.763754
306,GOOGL,United States,3.636196e+12,3.704355
305,GOOG,United States,3.596241e+12,3.694644
42,AMZN,United States,2.883431e+12,3.500490
61,AVGO,United States,1.656497e+12,3.013340
421,META,United States,1.640314e+12,3.004711
626,TSLA,United States,1.542445e+12,2.950643
394,LLY,United States,1.285790e+12,2.790688


## 7. Checks

- Correlation between factors: they should be distinct (|ρ| well below 0.5).
- Top and bottom names per factor, to sanity-check directions.

In [12]:
SCORES = ["momentum", "quality", "value", "growth", "size"]
df[SCORES].corr().round(2)

,momentum,quality,value,growth,size
momentum,1.00,0.07,-0.16,0.14,0.11
quality,0.07,1.00,-0.21,-0.03,0.21
value,-0.16,-0.21,1.00,-0.07,-0.30
growth,0.14,-0.03,-0.07,1.00,0.04
size,0.11,0.21,-0.30,0.04,1.00


In [13]:
for factor in SCORES:
    ranked = df.sort_values(factor, ascending=False)
    top = ", ".join(ranked["symbol"].head(5))
    bottom = ", ".join(ranked["symbol"].tail(5))
    print(f"{factor:9s} top: {top}\n{'':9s} bottom: {bottom}")

momentum  top: OKTA, MX.TO, SWK, AYA.TO, MRNA
          bottom: BSX, ALB, NRG, COIN, FIS
quality   top: MA, V, NEM, APP, SNDK
          bottom: MRNA, IRM, BHC.TO, FDXF, SKE.TO
value     top: SMCI, UAL, APTV, PCG, BBUC.TO
          bottom: BEP-UN.TO, NXE.TO, DML.TO, NGEX.TO, MRNA
growth    top: BE, LITE, GPN, WAT, VMRK
          bottom: CPX.TO, MOS, EQB.TO, EIX, APO
size      top: AAPL, NVDA, MSFT, GOOGL, GOOG
          bottom: NCLH, RNO.PA, APTV, ARE, MOS


In [14]:
# Example cell: US Information Technology
cols = ["symbol"] + SCORES + ["mom_12m", "qual_roa", "val_fwd_ep", "gro_revenue"]
df[(df["market"] == "United States") & (df["sector"] == "Information Technology")][cols] \
    .sort_values("quality", ascending=False).round(2).head(10)

,symbol,momentum,quality,value,growth,size,mom_12m,qual_roa,val_fwd_ep,gro_revenue
700,APP,-0.85,3.10,-0.70,0.05,0.81,-24.14,0.37,0.05,0.56
716,SNDK,0.32,2.83,2.04,3.01,0.75,127.08,0.37,0.21,1.12
446,MU,1.55,2.46,1.62,1.99,2.46,127.08,0.37,0.21,1.12
469,NVDA,-1.47,2.04,-0.20,0.90,3.77,-12.73,0.37,0.10,1.06
586,STX,0.17,1.49,-0.07,0.71,0.72,102.12,0.37,0.12,0.61
509,PLTR,-0.19,1.39,-1.29,0.73,1.77,-1.90,0.19,0.02,0.96
656,VRSN,0.41,1.35,-1.07,-0.74,-0.57,25.11,0.37,0.04,0.06
61,AVGO,-0.70,1.24,-0.71,0.82,3.01,1.27,0.15,0.06,0.86
12,ADBE,-0.97,1.17,0.63,-0.55,0.50,-25.22,0.20,0.11,0.13
668,WDC,-0.35,1.12,0.24,0.03,0.64,81.99,0.26,0.14,0.56


## 8. Save

One row per stock: `ticker, asset_class, instrument_type` and the 5 scores. Saved locally and to
`s3://monport-datasets-381491955506/factors/factor_scores.parquet`.

In [15]:
scores = df.rename(columns={"symbol": "ticker"})[["ticker", "asset_class", "instrument_type"] + SCORES]
scores.to_parquet("data/factor_scores.parquet", index=False)

buffer = io.BytesIO()
scores.to_parquet(buffer, index=False)
boto3.client("s3", region_name="us-east-1").put_object(
    Bucket=BUCKET, Key="factors/factor_scores.parquet", Body=buffer.getvalue())

print(scores.shape)
scores.head()

(721, 8)


/workshop/.venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,ticker,asset_class,instrument_type,momentum,quality,value,growth,size
0,A,Equity,Stock,0.316612,0.392534,-0.823540,-0.403838,-0.080021
1,AAPL,Equity,Stock,0.704236,0.984502,-0.945722,-0.646425,4.014922
2,ABBV,Equity,Stock,0.254759,-0.002278,-0.751667,3.612390,1.997556
3,ABNB,Equity,Stock,1.763955,1.155051,-0.958811,0.897429,0.641380
4,ABT,Equity,Stock,-1.089677,-0.184684,-0.292625,-0.061462,1.100617
